<a href="https://colab.research.google.com/github/dutta-bikram/Titanic--Machine-Learning-from-Disaster/blob/main/Titanic_MachineLearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Load Data

In [10]:
import pandas as pd

train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

train_df.head()
test_passenger_ids = test_df["PassengerId"]

Combine Train and Test

In [11]:
combined = pd.concat(
    [train_df.drop("Survived", axis=1),
     test_df],
    ignore_index=True
)

Preprocess

In [12]:
combined["Age"] = combined["Age"].fillna(
    combined["Age"].median()
)
combined["Fare"] = combined["Fare"].fillna(
    combined["Fare"].median()
)
combined["Embarked"] = combined["Embarked"].fillna(
    combined["Embarked"].mode()[0]
)


from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
combined["Sex"] = le.fit_transform(
    combined["Sex"]
)
combined["Embarked"] = le.fit_transform(
    combined["Embarked"]
)


combined = combined.drop(
    ["PassengerId",
     "Name",
     "Ticket",
     "Cabin"],
    axis=1
)

Recover Train/Test

In [13]:
X = combined.iloc[:len(train_df)]
X_test = combined.iloc[len(train_df):]

y = train_df["Survived"]

Create Validation Set

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

Train Many Models

In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

In [16]:
models = {
    "Logistic Regression":
        LogisticRegression(max_iter=1000),

    "Decision Tree":
        DecisionTreeClassifier(max_depth=5),

    "Random Forest":
        RandomForestClassifier(
            n_estimators=100,
            random_state=42
        ),

    "SVM":
        SVC(),

    "KNN":
        KNeighborsClassifier()
}

Compare Models

In [17]:
from sklearn.metrics import accuracy_score

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    acc = accuracy_score(
        y_val,
        preds
    )
    results.append((name, acc))
    print(name, ":", acc)

Logistic Regression : 0.8100558659217877
Decision Tree : 0.7988826815642458
Random Forest : 0.8212290502793296
SVM : 0.659217877094972
KNN : 0.7039106145251397


Pick Best Model

In [18]:
best_model_name = max(
    results,
    key=lambda x: x[1]
)[0]

print(best_model_name)

Random Forest


Retrain on ALL Training Data

In [19]:
best_model.fit(X, y)

RandomForestClassifier(random_state=42)

Predict Test Data

In [20]:
test_predictions = best_model.predict(
    X_test
)

Create Submission

In [21]:
submission = pd.DataFrame({
    "PassengerId": test_passenger_ids,
    "Survived": test_predictions
})

submission.to_csv(
    "submission.csv",
    index=False
)